# Check data present in the embedded wsi
Use label_data as a comparison for the wsi embedded:
Check for missing patients_ids in wsi embedded that are present in label data,
check for missing slide_ids between the two datasets

In [1]:
import os

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

from zoorvival.data import load_tcga_data
from zoorvival.nn.training import as_torch_dataset

%reload_ext autoreload
%autoreload 2

np.random.seed(42)

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-kxdwtjrw because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
PROJECT = "BRCA"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### utils

In [4]:
def request_tcga_file_info(data_type: str, project_ids=None) -> pd.DataFrame:
    """
    Get TCGA file information from GDC API.

    Args:
        data_type (str): The type of data to retrieve.
        project_ids (list[str], optional): List of project IDs to filter by. Defaults to None.
    
    Returns:
        pd.DataFrame: DataFrame containing file information.
    """
    if project_ids is None:
        project_ids = get_tcga_projects()
    fields = [
        "file_name",
        "md5sum",
        "cases.submitter_id",
        "cases.samples.sample_type",
        "cases.project.project_id",
        "cases.project.primary_site",
    ]
    fields = ",".join(fields)
    files_endpt = "https://api.gdc.cancer.gov/files"
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {
                    "field": "files.experimental_strategy",
                    "value": [data_type],
                },
            },
            {
                "op": "in",
                "content": {
                    "field": "cases.project.project_id",
                    "value": project_ids,
                },
            },
        ],
    }
    params = {"filters": filters, "fields": fields, "format": "TSV", "size": "200000"}
    response = requests.post(
        files_endpt, headers={"Content-Type": "application/json"}, json=params
    )
    return pd.read_csv(StringIO(response.content.decode("utf-8")), sep="\t")


def process_tcga_file_info(df_files, suffix, data_type) -> pd.DataFrame:
    rename_cols = {
        "cases.0.project.project_id": "project_id",
        "cases.0.project.primary_site": "primary_site",
        "cases.0.submitter_id": "submitter_id",
        "cases.0.samples.0.sample_type": "sample_type",
        "file_name": f"{data_type}_file_name",
        "id": f"{data_type}_file_id",
        "md5sum": f"{data_type}_md5sum",
    }
    df_files = df_files.rename(columns=rename_cols)
    df_files = df_files[df_files[f"{data_type}_file_name"].str.endswith(suffix)]
    df_files = df_files[df_files["sample_type"] == "Primary Tumor"]
    df_files = df_files[~df_files.duplicated(subset=["submitter_id"], keep="first")]
    return df_files


def get_suffix_counts(df_files) -> pd.Series:
    file_col = [c for c in df_files.columns if "file_name" in c][0]
    file_suffixes = [".".join(f.split(".")[-2:]) for f in df_files[file_col]]
    suffix_counts = pd.Series(file_suffixes).value_counts()
    suffix_counts = suffix_counts[suffix_counts > 1]
    suffix_counts = suffix_counts.reset_index()
    return suffix_counts

def get_tcga_projects() -> list[str]:
    """
    Get a list of TCGA projects from the GDC API.
    """
    response = requests.get("https://api.gdc.cancer.gov/projects", params={"size": 10000})
    response.raise_for_status()
    projects = response.json()["data"]["hits"]
    projects = [proj for proj in projects if proj.get("project_id", "").startswith("TCGA-")]
    projects = [proj["id"] for proj in projects]
    return projects

## Check for missing patients_ids

get dataset already embedded

In [5]:
db = load_tcga_data(PROJECT)

In [6]:
db.train.wsi_embeddings.shape  

(675, 64, 1536)

In [7]:
db.train.df_clinical.shape

(675, 565)

In [8]:
db.train.df_clinical.head(5)

,age,her2_and_cent17_cells_count,her2_cent17_ratio,her2_ihc_score,ihc_score,initial_pathologic_dx_year,lymph_nodes_examined_he_count,lymph_nodes_examined_ihc_count,lymph_node_examined_count,patient_id___RARE__,...,tissue_source_site_C8,tissue_source_site_D8,tissue_source_site_E2,tissue_source_site_E9,tissue_source_site_EW,tissue_source_site_GM,tissue_source_site_LL,tissue_source_site_OL,tissue_source_site___RARE__,tumor_status_WITH TUMOR
A2-A0SX,-0.807573,-0.11475,-0.132887,-0.429025,0.179883,-0.202800,-0.504802,-0.194507,-1.159306,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
AQ-A04J,-1.033436,-0.11475,-0.132887,-1.849729,-6.426715,0.268747,-0.504802,-0.194507,1.165059,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
A8-A07G,0.472319,-0.11475,-0.132887,-0.429025,0.179883,0.268747,-0.265295,-0.194507,-0.180626,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AO-A0J6,0.171168,-0.11475,-0.132887,-1.849729,0.179883,0.032973,-0.504802,-0.194507,-0.792301,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AQ-A54N,-0.581710,-0.11475,-0.187358,0.991680,0.179883,0.740294,-0.504802,-0.194507,-1.159306,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [9]:
db.train.wsi_embeddings[:5]

array([[[ 0.37793252, -0.09149176,  0.6140099 , ..., -0.6362666 ,
          0.6590168 ,  0.07718655],
        [-0.42122453,  0.75855935, -0.27535447, ...,  0.65027934,
          0.90929216,  0.2585132 ],
        [-0.12001964,  0.45504895,  0.4268396 , ..., -0.3881944 ,
         -0.47757852, -0.43103996],
        ...,
        [ 0.131354  , -0.02943878,  0.9171525 , ...,  0.4913377 ,
         -0.16327634,  0.0373682 ],
        [ 0.9467    , -0.9050123 ,  0.690313  , ..., -0.9909107 ,
          0.21412738,  0.776048  ],
        [-1.5100002 , -2.0749156 ,  0.10983063, ...,  1.2128549 ,
         -1.0097764 ,  1.1125333 ]],

       [[-0.18137573,  0.2914176 , -0.03264935, ...,  0.9376054 ,
          1.9608705 , -0.6172608 ],
        [ 0.57427347, -0.5286775 ,  1.5157274 , ..., -0.51070786,
          0.14952697,  0.6765116 ],
        [-0.73249793, -0.42794076,  1.4650383 , ..., -0.5084935 ,
          0.7214007 ,  0.05868489],
        ...,
        [-0.48301625, -0.79125404,  1.265502  , ...,  

get label_data patients_ids

In [10]:
csv_file = "src/datasets_csv/metadata/tcga_brca.csv"
csv_df_patients = pd.read_csv(csv_file, usecols=["case_id"])
set_csv_patients = set(csv_df_patients["case_id"].astype(str))
print(f'Number of patients in CSV file: {len(set_csv_patients)}')
print(f'Patients in CSV file: {list(set_csv_patients)[:5]}')

Number of patients in CSV file: 871
Patients in CSV file: ['TCGA-WT-AB41', 'TCGA-AO-A0JE', 'TCGA-A8-A06U', 'TCGA-E9-A247', 'TCGA-A8-A08I']


get clinical data patients_ids

In [11]:
db.train.df_clinical.index

Index(['A2-A0SX', 'AQ-A04J', 'A8-A07G', 'AO-A0J6', 'AQ-A54N', 'B6-A0I1',
       'AO-A0J7', '5T-A9QA', 'E2-A10B', 'AC-A3QP',
       ...
       'A8-A09Q', 'BH-A0BS', 'A7-A13G', 'AR-A1AM', 'BH-A0BV', 'BH-A18R',
       'A7-A6VX', 'A2-A04P', 'A8-A082', 'E2-A1BD'],
      dtype='object', length=675)

In [12]:
train_patients = set('TCGA-' + i for i in db.train.df_clinical.index.astype(str))
# print first 5 elements of train_patients
print(f'First 5 training patients: {list(train_patients)[:5]}')
test_patients = set('TCGA-' + i for i in db.test.df_clinical.index.astype(str))
print(f'Number of patients in training set: {len(train_patients)}')
print(f'Number of patients in test set: {len(test_patients)}')
db_patients = train_patients.union(test_patients)
print(f'Number of patients in final set: {len(db_patients)}')

First 5 training patients: ['TCGA-C8-A8HQ', 'TCGA-C8-A12K', 'TCGA-AC-A3EH', 'TCGA-AO-A0JE', 'TCGA-B6-A0RP']
Number of patients in training set: 675
Number of patients in test set: 170
Number of patients in final set: 845


comparison

In [13]:
missing = set_csv_patients.difference(db_patients)
print(f'Number of missing WSI IDs: {len(missing)}')
if len(missing) > 0:
    print(f'Missing WSI IDs: {missing}\n')

Number of missing WSI IDs: 163
Missing WSI IDs: {'TCGA-EW-A6SD', 'TCGA-B6-A0X1', 'TCGA-GM-A2DB', 'TCGA-WT-AB41', 'TCGA-HN-A2NL', 'TCGA-EW-A1P0', 'TCGA-E9-A247', 'TCGA-EW-A1J5', 'TCGA-E9-A243', 'TCGA-EW-A1PH', 'TCGA-LD-A66U', 'TCGA-S3-AA14', 'TCGA-LL-A5YP', 'TCGA-OL-A5D7', 'TCGA-A2-A0ST', 'TCGA-PL-A8LX', 'TCGA-GM-A2DN', 'TCGA-GM-A2DO', 'TCGA-E9-A229', 'TCGA-S3-AA15', 'TCGA-A2-A0CX', 'TCGA-D8-A1JM', 'TCGA-LL-A6FP', 'TCGA-EW-A423', 'TCGA-LL-A6FR', 'TCGA-E9-A249', 'TCGA-A2-A0YE', 'TCGA-UU-A93S', 'TCGA-E9-A3X8', 'TCGA-XX-A89A', 'TCGA-E9-A22H', 'TCGA-A8-A06X', 'TCGA-E9-A22D', 'TCGA-E9-A248', 'TCGA-PE-A5DD', 'TCGA-PE-A5DC', 'TCGA-EW-A1OZ', 'TCGA-BH-A1FR', 'TCGA-GM-A2DF', 'TCGA-GM-A3NW', 'TCGA-EW-A2FW', 'TCGA-A8-A08P', 'TCGA-AO-A1KO', 'TCGA-GM-A2DK', 'TCGA-S3-AA10', 'TCGA-AR-A24T', 'TCGA-EW-A1J1', 'TCGA-GI-A2C8', 'TCGA-GM-A2DD', 'TCGA-EW-A1P8', 'TCGA-E9-A22A', 'TCGA-EW-A1PA', 'TCGA-OL-A5D8', 'TCGA-OL-A6VO', 'TCGA-5L-AAT1', 'TCGA-E9-A244', 'TCGA-S3-AA0Z', 'TCGA-OL-A5DA', 'TCGA-GM-A2D9', 'TCGA-G

In [14]:
found = db.train.df_clinical[db.train.df_clinical.index.isin(db_patients)]
print(f'Number of found WSI IDs: {len(found)}')

Number of found WSI IDs: 0


In [15]:
wsi_train = ( db.train.df_clinical.index == 'E9-A228').any()
print(f'wsi found: {wsi_train}')
wsi_test = ( db.test.df_clinical.index == 'E9-A228').any()
print(f'wsi found: {wsi_test}')

wsi found: False
wsi found: False


## test function

In [16]:
slide_ids = ['TCGA-5L-AAT1-01Z-00-DX1.F3449A5B-2AC4-4ED7-BF44-4C8946CDB47D.svs', 'TCGA-A1-A0SI-01Z-00-DX1.AB717348-F964-4F29-BBE2-972B7C640432.svs', 'TCGA-A1-A0SQ-01Z-00-DX1.36071264-3407-4224-BCBB-2ED00294569A.svs']

In [18]:
data_wsi = load_tcga_data(PROJECT)
patch_features = []

print('getting the WSI embeddings for the slides')
zoo_slides = []
# get the right format for the slide_ids: can find them in zoorvival using a short version of patient_id
for slide in slide_ids:
    id = slide[5:12]   # remove 'TCGA-' and keep the case_id
    zoo_slides.append(id)

# load all slide_ids corresponding for the patient
for id in zoo_slides:
    wsi_bag = np.zeros((64, 1536))
    #wsi_path = os.path.join(data_wsi, '{}.pt'.format(slide_id.rstrip('.svs')))
    #wsi_bag = torch.load(wsi_path)

    # chec if id is a index in train or test separation for df_clinical
    if id in data_wsi.train.df_clinical.index:
        wsi_idx = data_wsi.train.df_clinical.index.get_loc(id)   # get the integer index from the string index
        wsi_bag = data_wsi.train.wsi_embeddings[wsi_idx]  # get the wsi bag for this index
    elif id in data_wsi.test.df_clinical.index:
        wsi_idx = data_wsi.test.df_clinical.index.get_loc(id)
        wsi_bag = data_wsi.test.wsi_embeddings[wsi_idx] 
    else:
        print("PROBLEM! Slide ID {} not found in either train or test dataset.".format(id))
        # go to next slide_id skipping this iteration
        continue  # skip to the next slide_id

    # convert numpy array to torch tensor
    if isinstance(wsi_bag, np.ndarray):
        if wsi_bag.ndim == 1:
            wsi_bag = wsi_bag.reshape(1, -1)
    
    wsi_bag = torch.from_numpy(wsi_bag)
    patch_features.append(wsi_bag)

patch_features = torch.cat(patch_features, dim=0)

print(f'Number of patches: {patch_features.shape}')
print(f'Print patches: \n{patch_features}')

getting the WSI embeddings for the slides
PROBLEM! Slide ID 5L-AAT1 not found in either train or test dataset.
Number of patches: torch.Size([128, 1536])
Print patches: 
tensor([[ 0.0606, -0.5646,  0.2842,  ..., -1.0158, -0.1464,  0.6940],
        [-0.9272, -0.1530,  0.6257,  ..., -1.0758,  0.4775,  0.0606],
        [ 0.1114,  0.3176,  0.1099,  ..., -0.6534, -0.2376, -0.3182],
        ...,
        [-0.8177,  0.2577,  0.4964,  ..., -1.3248,  0.4955,  0.0643],
        [-0.1715, -0.4413,  0.2454,  ..., -0.0957,  0.2791,  0.1078],
        [-0.6131, -0.3695,  0.0946,  ..., -1.3610,  0.0098, -0.3443]])


## get raw data to check for missing slide_ids
in the zoorvival dataset all patients have only one slide each, to understand which slide it corresponds to in the csv datatsets, it was used the raw version of the wsi data that contains the slide_ids

In [18]:
import random
import requests
from io import StringIO
from loguru import logger
import seaborn as sns

In [19]:
WSI_files = request_tcga_file_info(data_type="Diagnostic Slide")

print(f'shape of file: {WSI_files.shape}')
print(f'columns of WSI files: {WSI_files.columns.tolist()}')
print(f'first five of WSI files:\n{WSI_files.head()}')

print(f'suffix count: {get_suffix_counts(WSI_files)}')

df_WSI = process_tcga_file_info(WSI_files, suffix=".svs", data_type="WSI")
print(f'shape of processed WSI files: {df_WSI.shape}')
print(f'number of unique WSI IDs: {df_WSI["submitter_id"].nunique()}')
print(f'first five WSI IDs:\n{df_WSI["submitter_id"].head()}')
print(f'columns of processed WSI files: {df_WSI.columns.tolist()}')
print(f'first five WSI files:\n{df_WSI.head()}')

print(f'number of unique patients: {df_WSI["submitter_id"].nunique()}')


shape of file: (11766, 8)
columns of WSI files: ['cases.0.project.primary_site', 'cases.0.project.project_id', 'cases.0.samples.0.sample_type', 'cases.0.samples.1.sample_type', 'cases.0.submitter_id', 'file_name', 'id', 'md5sum']
first five of WSI files:
  cases.0.project.primary_site cases.0.project.project_id  \
0                       Breast                  TCGA-BRCA   
1                       Breast                  TCGA-BRCA   
2                       Breast                  TCGA-BRCA   
3                       Breast                  TCGA-BRCA   
4                       Breast                  TCGA-BRCA   

  cases.0.samples.0.sample_type cases.0.samples.1.sample_type  \
0                 Primary Tumor                           NaN   
1                 Primary Tumor                           NaN   
2                 Primary Tumor                           NaN   
3                 Primary Tumor                           NaN   
4                 Primary Tumor                      

### comparison of patient ids

get label dataset and patient ids from the dataset

In [21]:
csv_file = "src/datasets_csv/metadata/tcga_brca.csv"
csv_df_patients = pd.read_csv(csv_file, usecols=["case_id"])
set_csv_patients = set(csv_df_patients["case_id"].astype(str))
print(f'Number of patients in CSV file: {len(set_csv_patients)}')
print(f'Patients in CSV file: {list(set_csv_patients)[:5]}')

Number of patients in CSV file: 871
Patients in CSV file: ['TCGA-AN-A0FK', 'TCGA-LL-A73Y', 'TCGA-A8-A085', 'TCGA-AO-A0JE', 'TCGA-E2-A15K']


get patient ids from wsi dataset

In [22]:
#get patient ids from the WSI files
wsi_ids_patients = set(df_WSI["submitter_id"].astype(str))
print(f'Number of patients in WSI files: {len(wsi_ids_patients)}')
print(f'Patients in WSI files: {list(wsi_ids_patients)[:5]}')

Number of patients in WSI files: 9498
Patients in WSI files: ['TCGA-AG-3581', 'TCGA-2Z-A9JP', 'TCGA-35-3615', 'TCGA-UB-A7MC', 'TCGA-78-7149']


In [23]:
missing = set_csv_patients - wsi_ids_patients
print(f'Number of missing WSI IDs: {len(missing)}')
if len(missing) > 0:
    print(f'Missing WSI IDs: {missing}\n')

Number of missing WSI IDs: 0


### comparison of slide ids

In [24]:
csv_df_slides = pd.read_csv(csv_file, usecols=["slide_id"])
set_csv_slides = set(csv_df_slides["slide_id"].astype(str))
print(f'Number of slides in CSV file: {len(set_csv_slides)}')
print(f'Slides in CSV file: {list(set_csv_slides)[:5]}')

#get patient ids from the WSI files
wsi_ids_slides = set(df_WSI["WSI_file_name"].astype(str))
print(f'Number of slides in WSI files: {len(wsi_ids_slides)}')
print(f'slides in WSI files: {list(wsi_ids_slides)[:5]}')

Number of slides in CSV file: 931
Slides in CSV file: ['TCGA-BH-A0EE-01Z-00-DX1.872EA586-25C8-4835-A138-152DFA1EBF30.svs', 'TCGA-A8-A08P-01Z-00-DX1.431A1290-9CF4-4CE7-A9F0-13F9ADA46ADA.svs', 'TCGA-D8-A1XZ-01Z-00-DX2.73D59546-C003-4DED-80D5-866E7055EC79.svs', 'TCGA-D8-A1X8-01Z-00-DX1.DDA3866E-C7CC-4650-BE2B-5C9A98A2D531.svs', 'TCGA-EW-A1J5-01Z-00-DX1.FFE303B1-7F36-4276-B931-C2A6B2CBD1F2.svs']
Number of slides in WSI files: 9498
slides in WSI files: ['TCGA-A3-3387-01Z-00-DX1.01a24641-ab38-41bc-9151-48e262eb4826.svs', 'TCGA-HW-7490-01Z-00-DX1.2109dc98-1288-4da2-963e-4544db680603.svs', 'TCGA-C8-A27A-01Z-00-DX1.0E26C46D-CD65-40F3-8976-EB4415582934.svs', 'TCGA-56-6545-01Z-00-DX1.24a6678b-2dbf-4539-844e-1e3786c3fac3.svs', 'TCGA-VQ-A922-01Z-00-DX1.445608A5-D4F1-44AB-8642-3C944D753D87.svs']


In [25]:
missing = set_csv_slides - wsi_ids_slides
print(f'Number of missing WSI IDs: {len(missing)}')  # 60
if len(missing) > 0:
    print(f'Missing WSI IDs: {missing}\n')

Number of missing WSI IDs: 60
Missing WSI IDs: {'TCGA-A7-A0CH-01Z-00-DX2.81DDF423-FCA8-46CA-83A4-2E80B611844D.svs', 'TCGA-D8-A1JE-01Z-00-DX1.714805A1-E337-46DA-88D9-6CE4B4E3C2D0.svs', 'TCGA-PL-A8LZ-01A-01-DX1.B9F233EE-06DD-4C14-A392-D2937C7C0868.svs', 'TCGA-D8-A1JD-01Z-00-DX1.6D215B14-DD90-4635-8645-AF06EBD9BA3F.svs', 'TCGA-D8-A1X7-01Z-00-DX2.F0631B8C-EB75-4995-8ED7-1A8972BE8997.svs', 'TCGA-D8-A27R-01Z-00-DX1.F6E2FD1C-0666-4788-8D95-A76D15907270.svs', 'TCGA-D8-A1JT-01Z-00-DX1.F278C419-E405-4BDA-BA50-BFBA08801168.svs', 'TCGA-PL-A8LY-01A-01-DX1.32047D5E-8A42-480A-B2B8-A56B47B949FD.svs', 'TCGA-A7-A0CD-01Z-00-DX1.F045B9C8-049C-41BF-8432-EF89F236D34D.svs', 'TCGA-OK-A5Q2-01Z-00-DX4.83B45D6C-E350-4436-812F-4155D9F7D331.svs', 'TCGA-OK-A5Q2-01Z-00-DX1.0D169898-37C6-44CA-AC87-27887123AA6F.svs', 'TCGA-D8-A27T-01Z-00-DX1.1E3A4D57-9CF2-4EBF-B74D-ADD7BD8CBFA5.svs', 'TCGA-3C-AALJ-01Z-00-DX1.777C0957-255A-42F0-9EEB-A3606BCF0C96.svs', 'TCGA-D8-A1XR-01Z-00-DX2.A103FB8B-4397-4DD4-8587-90A736407484.svs', 

### check if specific patients/slide_id are present

In [26]:
wsi_found = (df_WSI['WSI_file_name'] == 'TCGA-A2-A0SV-01Z-00-DX1.F5645E47-3540-4753-AF5D-F5709BD8DFC1.svs').any()
print(f'wsi found: {wsi_found}')
wsi = df_WSI[df_WSI['WSI_file_name'] == 'TCGA-A2-A0SV-01Z-00-DX1.F5645E47-3540-4753-AF5D-F5709BD8DFC1.svs']
print(f'wsi info:\n{wsi}') # found

wsi found: True
wsi info:
      primary_site project_id    sample_type cases.0.samples.1.sample_type  \
10883       Breast  TCGA-BRCA  Primary Tumor                           NaN   

       submitter_id                                      WSI_file_name  \
10883  TCGA-A2-A0SV  TCGA-A2-A0SV-01Z-00-DX1.F5645E47-3540-4753-AF5...   

                                WSI_file_id                        WSI_md5sum  
10883  416e4645-4ef2-4228-8e93-703d138834e4  0af6063ab09eecbd76ecc6896e5a8f88  


In [27]:
wsi_found = (df_WSI['WSI_file_name'] == 'TCGA-D8-A27R-01Z-00-DX1.F6E2FD1C-0666-4788-8D95-A76D15907270.svs').any()
print(f'wsi found: {wsi_found}')
wsi = df_WSI[df_WSI['WSI_file_name'] == 'TCGA-D8-A27R-01Z-00-DX1.F6E2FD1C-0666-4788-8D95-A76D15907270.svs']
print(f'wsi info:\n{wsi}') # not found

wsi found: False
wsi info:
Empty DataFrame
Columns: [primary_site, project_id, sample_type, cases.0.samples.1.sample_type, submitter_id, WSI_file_name, WSI_file_id, WSI_md5sum]
Index: []


In [28]:
wsi_found = (df_WSI['submitter_id'] == 'TCGA-D8-A27R').any()
print(f'wsi found: {wsi_found}')
wsi = df_WSI[df_WSI['submitter_id'] == 'TCGA-D8-A27R']
print(f'wsi info:\n{wsi}') # found
print(f'wsi file name: {wsi["WSI_file_name"]}')  # TCGA-D8-A27R-01Z-00-DX2.31F47D8F-DFD7-42AE-BBB...

wsi found: True
wsi info:
      primary_site project_id    sample_type cases.0.samples.1.sample_type  \
11136       Breast  TCGA-BRCA  Primary Tumor                           NaN   

       submitter_id                                      WSI_file_name  \
11136  TCGA-D8-A27R  TCGA-D8-A27R-01Z-00-DX2.31F47D8F-DFD7-42AE-BBB...   

                                WSI_file_id                        WSI_md5sum  
11136  6f29dfa9-d166-4f07-9cdb-0daceb7341e9  8f47d35be3f649ce444fc88774b18b33  
wsi file name: 11136    TCGA-D8-A27R-01Z-00-DX2.31F47D8F-DFD7-42AE-BBB...
Name: WSI_file_name, dtype: object
